# 5200 Final Exam — Mock Set 2 (with Answer Code)
## Dataset: Mr Beast YouTube Videos (`mr_beast.csv`)

### Exam Rules
- **Time**: 1 hr 50 min | `alpha = 0.05` | `random_state = 617`
- Do NOT round | No commas | Lead with 0 | Open-book

### Modules Covered
Non-Linear Regression · Feature Selection · Trees · Ensemble Models · SVM

### Variables
| Variable | Description |
|---|---|
| `likes` | Number of likes (**regression outcome**) |
| `viral` | Created variable: 1 if likes>1M (**classification outcome**) |
| `views` | Number of views |
| `comments` | Number of comments |
| `duration_seconds` | Video length |
| `number_of_tags` | Tags count |
| `length_description` | Description length |
| `length_title` | Title length |
| `time_since` | Hours since upload |
| `publish_day` | Day of week |

---
## ❓ Q1. [Dataset Confirmation]
> Read in `mr_beast.csv`. How many rows are in the dataset?

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('mr_beast.csv')
data['publish_day'] = pd.Categorical(data['publish_day'],
    categories=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], ordered=True)

print('Rows:', data.shape[0])
print('Shape:', data.shape)

---
## ❓ Q2. [Setup]
> Create `viral = 1 if likes > 1,000,000 else 0`.
> Features: `views, comments, duration_seconds, number_of_tags, length_description, length_title, time_since, publish_day`.
> Stratified 80/20 split using `y_binned = pd.cut(y, bins=[-1,0,1])`.
> **What is the percentage of viral videos in the TRAIN sample?**

In [ ]:
from sklearn.model_selection import train_test_split

data['viral'] = (data.likes > 1000000).astype(int)

y = data.viral
X = data[['views','comments','duration_seconds','number_of_tags',
          'length_description','length_title','time_since','publish_day']]

y_binned = pd.cut(y, bins=[-1, 0, 1])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=617, stratify=y_binned)

print('% viral in train:', y_train.mean() * 100)

---
## Section 1: Non-Linear Regression
*(Predict `likes` as numeric. Use a separate 90/10 split with `data.sample()`.)*

---
## ❓ Q3.
> Split data 90/10 with `data.sample(frac=0.9, random_state=617)` for likes prediction.
> Fit:
> - `model_lin`: `likes ~ time_since`
> - `model_poly2`: `likes ~ time_since + I(time_since**2)`
>
> **What is the R² of `model_poly2`?**

In [ ]:
from statsmodels.formula.api import ols

train_reg = data.sample(frac=0.9, random_state=617)
test_reg  = data.drop(labels=train_reg.index)

model_lin   = ols('likes ~ time_since', data=train_reg).fit()
model_poly2 = ols('likes ~ time_since + I(time_since**2)', data=train_reg).fit()
model_poly3 = ols('likes ~ time_since + I(time_since**2) + I(time_since**3)', data=train_reg).fit()

print('Linear   R²:', model_lin.rsquared)
print('Poly-2   R²:', model_poly2.rsquared)
print('Poly-3   R²:', model_poly3.rsquared)

---
## ❓ Q4.
> Based on `model_poly2`, is the quadratic term (`I(time_since**2)`) statistically significant?
> **What is its p-value?**

In [ ]:
print(model_poly2.summary())
print('\np-value of quadratic term:', model_poly2.pvalues['I(time_since ** 2)'])
print('Significant:', model_poly2.pvalues['I(time_since ** 2)'] < 0.05)

---
## ❓ Q5.
> Create a step function:
> - Bins: `[0, 10000, 30000, 60000, inf)` → labels: `new, mid, old, very_old`
> - Fit: `likes ~ time_bin`
>
> **What is the R² of `model_step`?**

In [ ]:
train_reg = train_reg.copy()
test_reg  = test_reg.copy()

bins   = [0, 10000, 30000, 60000, float('inf')]
labels = ['new', 'mid', 'old', 'very_old']

train_reg['time_bin'] = pd.cut(train_reg.time_since, bins=bins, labels=labels)
test_reg['time_bin']  = pd.cut(test_reg.time_since,  bins=bins, labels=labels)

model_step = ols('likes ~ time_bin', data=train_reg).fit()
print('Step R²:', model_step.rsquared)
print('\nCoefficients (vs reference=new):'); print(model_step.params)

---
## ❓ Q6.
> Compare test RMSE: `model_lin` vs `model_poly2`.
> **What is the test RMSE of `model_poly2`?**

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

print('Linear  test RMSE:', rmse(test_reg.likes, model_lin.predict(test_reg)))
print('Poly-2  test RMSE:', rmse(test_reg.likes, model_poly2.predict(test_reg)))
print('Poly-3  test RMSE:', rmse(test_reg.likes, model_poly3.predict(test_reg)))
print('Step    test RMSE:', rmse(test_reg.likes, model_step.predict(test_reg)))

---
## Section 2: Feature Selection
*(Predict `likes` numeric. Use the 90/10 split. Numeric features only: `duration_seconds, number_of_tags, length_description, length_title, time_since`.)*

---
## ❓ Q7.
> Compute Pearson correlations with `likes` for each numeric feature.
> **Which variable has the WEAKEST (smallest |r|) correlation with likes?**

In [ ]:
from scipy.stats import pearsonr

num_feats = ['duration_seconds','number_of_tags','length_description','length_title','time_since']
X_tr_num = train_reg[num_feats]
y_tr_num = train_reg['likes']

results = []
for col in num_feats:
    r, p = pearsonr(train_reg[col], train_reg['likes'])
    results.append({'variable': col, 'r': r, '|r|': abs(r), 'p': p, 'sig': p < 0.05})
df_corr = pd.DataFrame(results).sort_values('|r|')
print(df_corr)
print('\nWeakest:', df_corr.iloc[0]['variable'])

---
## ❓ Q8.
> Run a multiple linear regression with ALL five numeric predictors (statsmodels).
> **Which predictors are statistically significant at alpha=0.05?**

In [ ]:
formula_all = 'likes ~ ' + ' + '.join(num_feats)
model_all = ols(formula_all, data=train_reg).fit()
print(model_all.summary())
print('\nSignificant predictors (p<0.05):')
print(model_all.pvalues[model_all.pvalues < 0.05])

---
## ❓ Q9.
> Compute the VIF for each numeric predictor.
> **Which predictors have VIF > 5?**

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_c = sm.add_constant(X_tr_num)
vif_values = [variance_inflation_factor(X_c.values, i) for i in range(X_c.shape[1])]
vif_df = pd.DataFrame({'feature': X_c.columns, 'VIF': vif_values})
print(vif_df)
print('\nVIF > 5:')
print(vif_df[vif_df.VIF > 5])

---
## ❓ Q10.
> Run **Forward Variable Selection** using `SequentialFeatureSelector`:
> - estimator = `LinearRegression()`
> - `k_features='best'`, `forward=True`, `scoring='r2'`, `cv=5`
>
> **Which variables are selected?**

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LinearRegression

# Forward selection
sfs_fwd = SFS(LinearRegression(), k_features='best',
              forward=True, floating=False, scoring='r2', cv=5)
sfs_fwd.fit(X_tr_num, y_tr_num)
print('Forward selected:', sfs_fwd.k_feature_names_)

---
## ❓ Q11.
> Standardize features with `StandardScaler`, then run `LassoCV(cv=5)`.
> **Which predictors have coefficients shrunk to zero?**

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

scaler_fs = StandardScaler()
X_tr_sc = scaler_fs.fit_transform(X_tr_num)
X_te_sc = scaler_fs.transform(test_reg[num_feats])

lasso = LassoCV(cv=5, random_state=617).fit(X_tr_sc, y_tr_num)
coefs = pd.Series(lasso.coef_, index=num_feats)
print('All Lasso coefficients:')
print(coefs)
print('\nZeroed out (== 0):')
print(coefs[coefs == 0].index.tolist())
print('\nSelected (non-zero):')
print(coefs[coefs != 0].index.tolist())

---
## ❓ Q12.
> Apply PCA. **How many components retain ≥ 60% of variance?**
> Fit a LinearRegression with those components. **What is the TRAIN R²?**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

pca_full = PCA().fit(X_tr_sc)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
print('Cumulative variance by component:')
for i, v in enumerate(cumvar, 1):
    print(f'  {i} components: {v:.4f}')

k = int(np.argmax(cumvar >= 0.60)) + 1
print(f'\nComponents for 60% variance: {k}')

pca_k = PCA(n_components=k)
X_tr_pca = pca_k.fit_transform(X_tr_sc)
X_te_pca = pca_k.transform(X_te_sc)

lr_pca = LinearRegression().fit(X_tr_pca, y_tr_num)
print('\nTrain R² (PCA components):', r2_score(y_tr_num, lr_pca.predict(X_tr_pca)))
print('Test  R² (PCA components):', r2_score(test_reg['likes'], lr_pca.predict(X_te_pca)))

---
## Section 3: Decision Trees
*(Predict `viral` (binary). Use the 80/20 split from Q2. Dummy-encode `publish_day`.)*

---
## ❓ Q13.
> Dummy-encode `publish_day` (`drop_first=True`). Align test to train columns.
> Fit a Classification Tree: ALL features, `max_depth=3, random_state=617`.
> **What is the TRAIN accuracy?**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_tr_cls = pd.get_dummies(X_train_raw, columns=['publish_day'], drop_first=True).astype(float)
X_te_cls = pd.get_dummies(X_test_raw,  columns=['publish_day'], drop_first=True).astype(float)
X_te_cls = X_te_cls.reindex(columns=X_tr_cls.columns, fill_value=0)

tree3 = DecisionTreeClassifier(max_depth=3, random_state=617)
tree3.fit(X_tr_cls, y_train)

print('Tree (depth=3) Train accuracy:', accuracy_score(y_train, tree3.predict(X_tr_cls)))

---
## ❓ Q14.
> For the `max_depth=3` tree, **what is the TEST accuracy?**
> Also fit a maximal tree (no constraints). Which has larger gap between train and test?

In [ ]:
# Depth-3 test accuracy
print('Depth-3 Test accuracy:', accuracy_score(y_test, tree3.predict(X_te_cls)))

# Maximal tree (no constraints)
tree_max = DecisionTreeClassifier(random_state=617)
tree_max.fit(X_tr_cls, y_train)
print('\nMaximal Tree Train accuracy:', accuracy_score(y_train, tree_max.predict(X_tr_cls)))
print('Maximal Tree Test accuracy: ', accuracy_score(y_test, tree_max.predict(X_te_cls)))
print('(Larger gap = more overfitting)')

---
## ❓ Q15.
> Tune with `GridSearchCV`, 5-fold CV:
> `max_depth=[2,4,6,8,10]`, `min_samples_leaf=[5,20,50]`.
> **What is the optimal combination?**

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_cls = GridSearchCV(
    DecisionTreeClassifier(random_state=617),
    {'max_depth': [2,4,6,8,10], 'min_samples_leaf': [5,20,50]},
    cv=5, n_jobs=-1
)
grid_cls.fit(X_tr_cls, y_train)
print('Best params:', grid_cls.best_params_)
print('Best CV score:', grid_cls.best_score_)

---
## ❓ Q16.
> **What is the TEST accuracy of the tuned model from Q15?**

In [ ]:
print('Tuned tree Train accuracy:', grid_cls.score(X_tr_cls, y_train))
print('Tuned tree Test accuracy: ', grid_cls.score(X_te_cls, y_test))

---
## Section 4: Ensemble Models
*(Predict `viral`. Use same split + dummy-encoded features.)*

---
## ❓ Q17.
> Fit a **Voting Classifier** combining `LogisticRegression(max_iter=1000)` and
> `DecisionTreeClassifier(max_depth=4)`. `random_state=617`.
> **What is the TRAIN accuracy?**

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression

voting = VotingClassifier(estimators=[
    ('lr',   LogisticRegression(max_iter=1000, random_state=617)),
    ('tree', DecisionTreeClassifier(max_depth=4, random_state=617))
])
voting.fit(X_tr_cls, y_train)

print('Voting Train accuracy:', voting.score(X_tr_cls, y_train))
print('Voting Test accuracy: ', voting.score(X_te_cls, y_test))

---
## ❓ Q18.
> Fit a **Random Forest** with 100 trees, `oob_score=True, random_state=617`.
> **What is the OOB (Out-of-Bag) score?**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=617)
rf.fit(X_tr_cls, y_train)

print('RF OOB score:    ', rf.oob_score_)
print('RF Train accuracy:', rf.score(X_tr_cls, y_train))
print('RF Test accuracy: ', rf.score(X_te_cls, y_test))

---
## ❓ Q19.
> Based on the Random Forest, **which feature is MOST IMPORTANT** for predicting viral status?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_tr_cls.columns)
print(importances.sort_values(ascending=False))
print('\nMost important:', importances.idxmax())

---
## ❓ Q20.
> Fit a **Gradient Boosting Classifier** with 100 trees, `random_state=617`.
> **What is the TEST accuracy?**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=617)
gb.fit(X_tr_cls, y_train)

print('GB Train accuracy:', gb.score(X_tr_cls, y_train))
print('GB Test accuracy: ', gb.score(X_te_cls, y_test))

---
## ❓ Q21.
> Fit a **Stochastic Gradient Boosting** with `n_estimators=100, max_features=0.5,
> subsample=0.7, random_state=617`.
> **Which feature is MOST IMPORTANT according to this model?**

In [ ]:
sgb = GradientBoostingClassifier(
    n_estimators=100, max_features=0.5, subsample=0.7, random_state=617)
sgb.fit(X_tr_cls, y_train)

sgb_imp = pd.Series(sgb.feature_importances_, index=X_tr_cls.columns)
print(sgb_imp.sort_values(ascending=False))
print('\nMost important (SGB):', sgb_imp.idxmax())

---
## Section 5: Support Vector Machines
*(Predict `viral`. Use numeric features only. Scale before fitting.)*

---
## ❓ Q22.
> Use numeric features: `views, comments, duration_seconds, number_of_tags,
> length_description, length_title, time_since`.
> Scale with `StandardScaler` (fit on train only).
> Fit `LinearSVC(C=1, random_state=617)`.
> **What is the TRAIN accuracy?**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

num_cols_svm = ['views','comments','duration_seconds','number_of_tags',
                'length_description','length_title','time_since']

X_svm_tr_raw = X_train_raw[num_cols_svm]
X_svm_te_raw = X_test_raw[num_cols_svm]

scaler_svm = StandardScaler()
X_svm_tr = scaler_svm.fit_transform(X_svm_tr_raw)  # fit ONLY on train!
X_svm_te = scaler_svm.transform(X_svm_te_raw)

svm_lin = LinearSVC(C=1, random_state=617)
svm_lin.fit(X_svm_tr, y_train)

print('LinearSVC Train accuracy:', svm_lin.score(X_svm_tr, y_train))
print('LinearSVC Test accuracy: ', svm_lin.score(X_svm_te, y_test))

---
## ❓ Q23.
> Tune an RBF SVM with `GridSearchCV`, 5-fold CV.
> `C=[0.1,1,10,100]`, `gamma=[0.001,0.01,0.1,1]`.
> **What is the best C and gamma?**

In [ ]:
grid_svm = GridSearchCV(
    SVC(kernel='rbf', random_state=617),
    {'C': [0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1]},
    cv=5, n_jobs=-1
)
grid_svm.fit(X_svm_tr, y_train)

print('Best params:   ', grid_svm.best_params_)
print('Best CV score: ', grid_svm.best_score_)

---
## ❓ Q24.
> Using the tuned SVM, **what is the TEST accuracy?**
> Compare tuned SVM vs Random Forest on the test set.

In [ ]:
print('Tuned SVM Test accuracy:', grid_svm.score(X_svm_te, y_test))
print('Random Forest Test acc: ', rf.score(X_te_cls, y_test))
print('\nBetter model:', 'SVM' if grid_svm.score(X_svm_te, y_test) > rf.score(X_te_cls, y_test) else 'Random Forest')

---
# 📋 Quick Reference — Code Patterns

## Non-Linear Regression
```python
# Polynomial (statsmodels) — MUST include all lower-degree terms
model_poly2 = ols('y ~ x + I(x**2)', data=train).fit()
model_poly3 = ols('y ~ x + I(x**2) + I(x**3)', data=train).fit()
# p-value of quadratic: model.pvalues['I(x ** 2)']

# Step function
train['bin'] = pd.cut(train.x, bins=[0,10000,30000,float('inf')], labels=['a','b','c'])
model_step = ols('y ~ bin', data=train).fit()
# Coefficient = difference from reference (first label)
```

## Feature Selection
```python
# Correlation
from scipy.stats import pearsonr
r, p = pearsonr(x, y)

# VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
X_c = sm.add_constant(X)
vif = [variance_inflation_factor(X_c.values, i) for i in range(X_c.shape[1])]
# VIF > 5 = concern; VIF > 10 = serious

# Sequential (mlxtend)
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
sfs = SFS(LinearRegression(), k_features='best',
          forward=True,   # True=Forward, False=Backward
          floating=False, # True=Stepwise
          scoring='r2', cv=5)
sfs.fit(X_train, y_train); print(sfs.k_feature_names_)

# Lasso
lasso = LassoCV(cv=5, random_state=617).fit(X_tr_sc, y_tr)
coefs[coefs == 0]   # zeroed = excluded

# PCA
pca = PCA().fit(X_tr_sc)
k = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.60) + 1
```

## ⚠️ Key Traps
| Trap | Rule |
|---|---|
| Polynomial lower terms | `I(x**2)` REQUIRES `x` in formula too |
| Step coefficient | = difference from FIRST label (reference) |
| VIF threshold | > 5 concern; > 10 serious |
| Lasso requires scaling | Always `StandardScaler` before Lasso |
| PCA % selection | `np.argmax(cumsum >= 0.60) + 1` |
| SVM requires scaling | `fit_transform` train, `transform` test only |
| OOB score | Set `oob_score=True` in RandomForest |
| Stochastic GB | Use `max_features` and `subsample` < 1.0 |